In [ ]:
import pandas as pd
fp = "../data/sba_loans_prepared/sba_loans_risk_good_train.csv"
df = pd.read_csv(fp)
sel_bad = df.RiskyNbrh == 1

In [ ]:
df_bb = df[sel_bad]

In [ ]:
cols = df_bb.columns.tolist()
cols = [c for c in cols if c not in ["RiskyNbrh", "LoanStatus"]]

In [ ]:
df_bb = df_bb[cols]

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import euclidean_distances
import numpy as np
import numpy.linalg as la

In [ ]:
start_number = .001
end_number = .01
num_values = 5

# Generate 10 evenly spaced values between start_number and end_number (inclusive)
window_sizes = np.linspace(start_number, end_number, num_values)

In [ ]:
X_dist = euclidean_distances(df_bb.values, df_bb.values)

In [ ]:
# Construct similarity matrix using a Gaussian kernel
num_conn_comp = {}
EPS = 0.01
for w in window_sizes:
    print(f"Now processing window size: {w}")
    gamma = w # Adjust sigma for desired width of similarity
    similarity_matrix = np.exp(- gamma * X_dist**2) 
    degree_matrix = np.diag(np.sum(similarity_matrix, axis=1))
    laplacian_matrix = degree_matrix - similarity_matrix
    eigvals, eigvecs = la.eig(laplacian_matrix)
    sorted_indices = np.argsort(eigvals)
    # Sort eigenvalues
    sorted_eigvals = eigvals[sorted_indices]
    
    # Sort eigenvectors by applying the same indices to the columns
    sorted_eigvecs = eigvecs[:, sorted_indices]

    sorted_eigvals[np.abs(sorted_eigvals) < EPS] = 0
    num_zero_eig_vals = np.sum(sorted_eigvals == 0)
    num_conn_comp[w] = num_zero_eig_vals

In [ ]:
df_sexp = pd.DataFrame.from_dict(num_conn_comp, orient="index").reset_index()
df_sexp.columns = ["sigma", "num_conn_comp"]

In [ ]:
df_sexp

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel

In [ ]:
import numpy as np


# Assuming 'distance_matrix' is your precomputed distance matrix
# You might need to adjust 'gamma' based on your data
affinity_matrix = rbf_kernel(X_dist, gamma=.001)

In [ ]:
from sklearn.manifold import SpectralEmbedding
embedding = SpectralEmbedding(n_components=4, affinity="precomputed")
X_transformed = embedding.fit_transform(affinity_matrix)

In [ ]:
df_sp = pd.DataFrame(X_transformed)
df_sp.columns = ["emb-"+str(i+1) for i in range(4)]

In [ ]:
import plotly.express as px
fig = px.scatter(df_sp, x="emb-1", y="emb-2")
fig.show()

In [ ]:
from sklearn.manifold import SpectralEmbedding
embedding = SpectralEmbedding(n_components=4, affinity="nearest_neighbors", n_neighbors=1)
X_transformed = embedding.fit_transform(df_bb.values)
df_sp = pd.DataFrame(X_transformed)
df_sp.columns = ["emb-"+str(i+1) for i in range(4)]

In [ ]:
fig = px.scatter(df_sp, x="emb-1", y="emb-2")
fig.show()

In [ ]:
fpb = "../data/sba_loans_prepared/sba_train_borr_info.csv"
dfbi = pd.read_csv(fpb)

In [ ]:
fpb = "../data/sba_loans_prepared/bad_borr_index.csv"
dfbbi = pd.read_csv(fpb)

In [ ]:
dfbb_info = dfbi[dfbi.index.isin(dfbbi.bad_borrower_index)]

In [ ]:
cols_needed = ["BorrCity", "BorrState", "BorrZip"]
dfbb_info = dfbb_info[cols_needed].reset_index(drop=True)

In [ ]:
dfbb_info